In [ ]:
import pandas as pd

from sklearn.metrics import roc_auc_score

from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score, precision_score

import seaborn as sns

import matplotlib.pyplot as plt

from sklearn.preprocessing import normalize


In [ ]:
import argparse

import pandas as pd

import lightgbm as lgb

import joblib

from sklearn.preprocessing import StandardScaler

import copy

               

import itertools

from sklearn.impute import SimpleImputer

import numpy as np


In [ ]:
def preprocess_data(df):

                    

                               

                                              

                                                       

                                                                        

    def _count_kmer(Dataset, k):               

        

                      

        dataset = copy.deepcopy(Dataset)

                                

        nucleotide = ['A', 'C', 'G', 'T']

        

                         

                  

        five = list(itertools.product(nucleotide, repeat=5))

        pentamer = [''.join(n) for n in five]

        

                  

        four = list(itertools.product(nucleotide, repeat=4))

        tetramer = [''.join(n) for n in four]

                 

        three = list(itertools.product(nucleotide, repeat=3))

        threemer = [''.join(n) for n in three]

        

                                                                  

        if k == 34:

            table_kmer = dict.fromkeys(threemer, 0)

            table_kmer.update(dict.fromkeys(tetramer, 0))

        elif k == 45:

            table_kmer = dict.fromkeys(tetramer, 0)

            table_kmer.update(dict.fromkeys(pentamer, 0))

        elif k == 345:

            table_kmer = dict.fromkeys(threemer, 0)

            table_kmer.update(dict.fromkeys(tetramer, 0))

            table_kmer.update(dict.fromkeys(pentamer, 0))

                                       

        for mer in table_kmer.keys():

            table_kmer[mer] = dataset["sequence"].apply(lambda x: x.count(mer))

        

                                                                           

        rawcount_kmer_df = pd.DataFrame(table_kmer)

        df1_rawcount = pd.concat([rawcount_kmer_df, dataset["name"]], axis=1)

        df1_rawcount.index = dataset["tag"]

                                                                        

        freq_kmer_df = rawcount_kmer_df.apply(lambda x: x / x.sum(), axis=1)

        df1 = pd.concat([freq_kmer_df, dataset["name"]], axis=1)

        df1.index = dataset["tag"]

        return df1, df1_rawcount

    df_kmer_test, df_kmer_test_raw = _count_kmer(df, 345)

    del df_kmer_test['name']

    x_kmer = df_kmer_test.values

    imputer = SimpleImputer(strategy='mean')

    x_test = imputer.fit_transform(x_kmer)

                                           

    return x_test


In [ ]:
def load_model(model_path):

    model = joblib.load(model_path)

    return model


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, average_precision_score, matthews_corrcoef

def predict(model, x_test, df):

    y_pred = model.predict(x_test)

    y_prob = model.predict_proba(x_test)[:,1]

    df['prediction'] = y_pred           

    df['prob'] = y_prob           

    evaluate_df = df

    auroc = roc_auc_score(evaluate_df["tag"], evaluate_df["prob"])

    auprc = average_precision_score(evaluate_df["tag"], evaluate_df["prob"])

    accuracy = accuracy_score(evaluate_df['tag'], evaluate_df['prediction'])

    f1 = f1_score(evaluate_df['tag'], evaluate_df['prediction'])

    mcc = matthews_corrcoef(evaluate_df['tag'], evaluate_df['prediction'])

    recall = recall_score(evaluate_df['tag'], evaluate_df['prediction'])

    precision = precision_score(evaluate_df['tag'], evaluate_df['prediction'])

    

    return evaluate_df


In [ ]:
def parse_fasta_to_dataframe(fasta_path):

    records = []

    with open(fasta_path, 'r') as file:

        name, tag, seq_lines = None, None, []

        for line in file:

            line = line.strip()

            if line.startswith('>'):

                if name:

                    sequence = ''.join(seq_lines)

                    records.append({'name': name, 'tag': tag, 'sequence': sequence})

                        

                header = line[1:]           

                                               

                if '_' in header:

                                  

                    last_underscore_index = header.rfind('_')

                    name = header[:last_underscore_index]                  

                    tag_str = header[last_underscore_index + 1:]                 

                    try:

                        tag = int(tag_str) 

                    except ValueError:

                        tag = None  

                else:

                    name = header

                    tag = None

                seq_lines = []

            else:

                seq_lines.append(line)

                  

        if name:

            sequence = ''.join(seq_lines)

            records.append({'name': name, 'tag': tag, 'sequence': sequence})

    

    return pd.DataFrame(records)



In [ ]:
model = load_model('circExor/archived/circExor_archived_2026_3/models/saved_models/circRNA_ML_Model_tridivided_Output/RandomForest/best_RandomForest_model.pkl')

df = parse_fasta_to_dataframe('circExor/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output/val_test_set_sequences.fasta')

x_test = preprocess_data(df)

evaluate_df = predict(model, x_test, df)



In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('All Species', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
                            

df_mmu = df[df['name'].str.startswith('mmu')].copy()

x_test = preprocess_data(df_mmu)

evaluate_df = predict(model, x_test, df_mmu)


In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('Mus Only', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
            

                    

min_count = df_mmu['tag'].value_counts().min()

                 

df_mmu_balanced = df_mmu.groupby('tag', group_keys=False).apply(lambda x: x.sample(n=min_count, random_state=42))

x_test = preprocess_data(df_mmu_balanced)

evaluate_df = predict(model, x_test, df_mmu_balanced)


In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('Mus Only', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
                 

model = load_model('circExor/cross_specises/corss_specises_output/RandomForest/best_RandomForest_model.pkl')

df = parse_fasta_to_dataframe('circExor/ML_models/circRNA_ML_Model_tridivided_intra5fold_Output/val_test_set_sequences.fasta')

x_test = preprocess_data(df)

evaluate_df = predict(model, x_test, df)



In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('All Species', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
             

x_test = preprocess_data(df_mmu)

evaluate_df = predict(model, x_test, df_mmu)


In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('Mus Only', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()


In [ ]:
          

x_test = preprocess_data(df_mmu_balanced)

evaluate_df = predict(model, x_test, df_mmu_balanced)


In [ ]:
cm = confusion_matrix(evaluate_df['tag'], evaluate_df['prediction'])

                   

plt.figure(figsize=(6, 5))

ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', 

                 xticklabels=['Celluar', 'EV'], yticklabels=['Celluar', 'EV'],

                 annot_kws={'size': 16, 'weight': 'bold'},                  

                 cbar_kws={'shrink': 0.8})                   

plt.title('Mus Only', fontsize=18, fontweight='bold')

plt.ylabel('Acurate label', fontsize=16)

plt.xlabel('Predicted label', fontsize=16)

plt.xticks(fontsize=15, fontweight='bold')

plt.yticks(fontsize=15, fontweight='bold')

                       

cbar = ax.collections[0].colorbar

cbar.ax.tick_params(labelsize=14)                      

plt.show()
